In [10]:
import random

GAME_TITLE = "GUESS THE NUMBER"
RESULTS_FILE = "game_results.txt"
EASY = ("Easy", 1, 20, 8)
MEDIUM = ("Medium", 1, 50, 7)
HARD = ("Hard", 1, 100, 6)
DIFFICULTIES = {
    "Easy": EASY,
    "Medium": MEDIUM,
    "Hard": HARD,
}
STARTING_POINTS = {
    "Easy": 50,
    "Medium": 100,
    "Hard": 150,
}
POINTS_FOR_REMAINING_ATTEMPT = 10
HINT_COST = 5
FIRST_GUESS_BONUS = 50
VALID_MENU_CHOICES = ["1","2", "3", "4", "5"]
#SECTION 2:Player
class Player:
    def __init__(self, name, points):
        self.name = name
        self.total_points = 0
        self.games_played = 0
        self.games_won = 0
        self.best_score = 0
        self.game_history = []
    def record_result(self, result):
        self.games_played += 1
        self.game_history.append(result)
        if result["won"]:
            self.games_won += 1
            self.total_points += result["score"]
        if result["score"] > self.best_score:
            self.best_score=result["score"]
    def get_win_rate(self):
        if self.games_played ==0:
            return 0.0
        rate = self.games_won / self.games_played * 100
        return rate
    def get_average_score(self):
        if self.games_played == 0:
            return 0.0
        average = self.total_points / self.games_played
        return average
    def show_summary(self):
        print()
        print("-"*40)
        print("PLAYER STATISTICS")
        print("-"*40)
        print("Player Name:", self.name)
        print("Games played:", self.games_played)
        print("Games won:", self.games_won)
        print("Games lost:", self.games_played - self.games_won)
        print("Total points:", self.total_points)
        print("Best score:", self.best_score)
        print("Win rate:", round(self.get_win_rate(), 1), "%")
        print("Average score:", round(self.get_average_score(), 1))
        print("-"*40)
    def show_history(self):
        print()
        print("GAME HISTORY")
        print("-"*40)
        if len(self.game_history) == 0:
            print("You have not played any games.")
            return
        for number, result in enumerate(self.game_history, start=1):
            if result["won"]:
                outcome = "Won"
            else:
                outcome = "Lost"
            print(
                number,
                "Difficulty:",result["difficulty"],
                "Result:", outcome,
                "Guesses:", result["guesses"],
                "Points:", result["score"],
            )
        print("-"*40)
#SECTION 3: DISPLAY
def show_header():
    print()
    print("="*50)
    print(GAME_TITLE.center(50))
    print("="*50)
    print("A number guessing game")
    print()
def show_menu():
    print()
    print("MAIN MENU")
    print("1. Play a game")
    print("2. View player statistics")
    print("3. View game history")
    print("4. Read instructions")
    print("5. Quit")
    choice = input("Enter your choice (1-5): ").strip()
    return choice
def show_instructions():
    print()
    print("HOW TO PLAY")
    print("-"*40)
    print("1. Choose a difficulty level")
    print("2. The computer selects a secret number.")
    print("3. Enter a guess within the displayed range.")
    print("4. The game tells you to guess higher or lower.")
    print("5. Try to guess the number before the attempts run out.")
    print("6. You can type HINT to receive a clue.")
    print("7. You can type QUIT to end the current game.")
    print("8. Winning games earn points.")
    print("9. Your results are saved in a text file.")
    print("-"*40)
def show_difficulty_menu():
    print()
    print("DIFFICULTY LEVELS")
    print("1. Easy: numbers 1-20, 8 attempts")
    print("2. Medium: numbers 1-50, 7 attempts")
    print("3. Hard: numbers 1-100, 6 attempts")
def show_round_start(difficulty, low, high, attempts):
    print()
    print("Round Start")
    print("-"*40)
    print("Difficulty:", difficulty)
    print("The number is between", low, "and", high)
    print("You have", attempts, "attempts")
    print("Type HINT to receive a clue or QUIT to end the current game.")
    print("-"*40)
def show_round_end(result):
        print()
        print("="*40)
        print("Round Summary")
        print("="*40)
        if result["won"]:
            print("You won!")
        else:
            print("You did not guess the number.")
        print("Secret number:", result["secret"])
        print("Difficulty:", result["difficulty"])
        print("Guesses Used:", result["guesses"])
        print("Hints used:", result["hints"])
        print("Points earned:", result["score"])
        print("="*40)
#SECTION 4: INPUTS
def get_player_name():
    while True:
        name = input("Enter your name: ").strip()
        if name =="":
            print("Your name cannot be blank.")
        elif len(name) > 25:
            print("Use 25 characters or fewer.")
        else:
            return name
def get_menu_choice():
    while True:
        choice = input("Choose an option (1-5): ").strip()
        if choice in VALID_MENU_CHOICES:
            return choice
        print("That is not a valid menu choice.")
        print("Please enter a number from 1 to 5.")
def get_difficulty():
    show_difficulty_menu()
    difficulty_choices = {
        "1": EASY,
        "2": MEDIUM,
        "3": HARD,
    }
    while True:
        choice = input("Select difficulty (1-3): ").strip()
        if choice in difficulty_choices:
            return difficulty_choices[choice]
        print("Invalid difficulty. Please enter 1, 2, or 3.")
def get_guess(low, high):
    while True:
        answer = input(
            "Enter a guess from " + str(low) + " to " + str(high) + ": "
        ).strip().lower()
        if answer == "hint":
            return "hint"
        if answer == "quit":
            return "quit"
        if answer == "":
            print("Input cannot be blank.")
            continue
        try:
            guess = int(answer)
        except ValueError:
            print("Invalid input. Please enter a whole number.")
            continue
        if guess < low or guess > high:
            print("Your guess must be between", low, "and", high)
            continue
        return guess
def ask_to_continue():
    while True:
        answer = input("Would you like to keep playing? (y/n): ")
        answer = answer.strip().lower()
        if answer == "y" or answer == "yes":
            return True
        if answer == "n" or answer == "no":
            return False
        print("Please enter y for yes or n for no.")
#SECTION 5: GAME FUNCTIONS
def choose_secret_number(low, high):
    secret = random.randint(low, high)
    return secret
def compare_guess(guess, secret):
    if guess < secret:
        return "low"
    if guess > secret:
        return "high"
    return "correct"
def display_feedback(feedback):
    if feedback == "low":
        print("Too low! Try a higher number.")
    elif feedback == "high":
        print("Too high! Try a lower number.")
    else:
        print("Correct! You found the secret number.")
def give_hint(secret):
    if secret % 2 == 0:
        print("Hint: the secret number is even.")
    else:
        print("Hint: the secret number is odd.")
    if secret <= 10:
        print("Hint: the secret number is 10 or less.")
    elif secret <= 25:
        print("Hint: the secret number is between 11 and 25.")
    else:
        print("Hint: the secret number is greater than 25.")
    print("Using a hint will reduce your score by", HINT_COST, "points.")
def calculate_score(
    won,
    difficulty,
    max_attempts,
    guesses_used,
    hints_used,
):
    if not won:
        return 0
    score = STARTING_POINTS[difficulty]
    attempts_left = max_attempts - guesses_used
    score = score + attempts_left * POINTS_FOR_REMAINING_ATTEMPT
    score = score - hints_used * HINT_COST
    if guesses_used == 1 and hints_used == 0:
        score = score + FIRST_GUESS_BONUS
    if score < 0:
        score = 0
    return score
def play_round(player):
    settings = get_difficulty()
    difficulty, low, high, max_attempts = settings
    secret = choose_secret_number(low, high)
    guesses_used = 0
    hints_used = 0
    won = False
    stopped_early = False
    show_round_start(difficulty, low, high, max_attempts)
    while guesses_used < max_attempts:
        attempts_left = max_attempts - guesses_used
        print("Attempts remaining:", attempts_left)
        guess = get_guess(low, high)
        if guess == "hint":
            hints_used += 1
            give_hint(secret)
            continue
        if guess == "quit":
            print("You chose to end the round.")
            stopped_early = True
            break
        guesses_used += 1
        feedback = compare_guess(guess, secret)
        display_feedback(feedback)
        if feedback == "correct":
            won = True
            break
    result = {
        "difficulty": difficulty,
        "secret": secret,
        "guesses": guesses_used,
        "hints": hints_used,
        "won": won,
        "score": calculate_score(
            won,
            difficulty,
            max_attempts,
            guesses_used,
            hints_used,
        ),
        "stopped_early": stopped_early,
    }
    player.record_result(result)
    show_round_end(result)
    save_result_to_file(player, result)
#SECTION 6: FILES
def save_result_to_file(player, result):
    try:
        with open(RESULTS_FILE, "a", encoding="utf-8") as file:
            if result["won"]:
                outcome = "Won"
            else:
                outcome = "Lost"
            file.write("Player: " + player.name + "\n")
            file.write("Difficulty: " + result["difficulty"] + "\n")
            file.write("Outcome: " + outcome + "\n")
            file.write("Guesses: " + str(result["guesses"]) + "\n")
            file.write("Hints: " + str(result["hints"]) + "\n")
            file.write("Score: " + str(result["score"]) + "\n")
            file.write("-" * 25 + "\n")
        print("The game result was saved.")
    except OSError:
        print("Warning: the result could not be saved.")
def show_saved_results():
    print()
    print("SAVED RESULTS")
    print("-" * 40)
    try:
        with open(RESULTS_FILE, "r", encoding="utf-8") as file:
            contents = file.read()
        if contents.strip() == "":
            print("The results file is empty.")
        else:
            print(contents)
    except FileNotFoundError:
        print("There are no saved results yet.")
    except OSError:
        print("The saved results file could not be opened.")
#SECTION 7: EXTRA STATS
def count_wins(player):
    wins = 0
    for result in player.game_history:
        if result["won"]:
            wins += 1
    return wins
def count_losses(player):
    losses = 0
    for result in player.game_history:
        if not result["won"]:
            losses += 1
    return losses
def get_highest_score(player):
    highest = 0
    for result in player.game_history:
        if result["score"] > highest:
            highest = result["score"]
    return highest
def get_total_hints(player):
    total = 0
    for result in player.game_history:
        total += result["hints"]
    return total
def get_average_guesses(player):
    if len(player.game_history) == 0:
        return 0.0
    total = 0
    for result in player.game_history:
        total += result["guesses"]
    return total / len(player.game_history)
def show_extra_statistics(player):
    print()
    print("EXTRA STATISTICS")
    print("-" * 40)
    print("Wins counted from history:", count_wins(player))
    print("Losses counted from history:", count_losses(player))
    print("Highest score:", get_highest_score(player))
    print("Total hints used:", get_total_hints(player))
    print("Average guesses:", round(get_average_guesses(player), 2))
    print("-" * 40)
def count_games_by_difficulty(player):
    counts = {
        "Easy": 0,
        "Medium": 0,
        "Hard": 0,
    }
    for result in player.game_history:
        difficulty = result["difficulty"]
        counts[difficulty] += 1
    return counts
def show_difficulty_statistics(player):
    counts = count_games_by_difficulty(player)
    print()
    print("GAMES BY DIFFICULTY")
    print("-" * 30)
    for difficulty, number in counts.items():
        print(difficulty + ":", number)
    print("-" * 30)
#SECTION 8: MENU
def show_summary(player):
    player.show_summary()
def view_statistics(player):
    show_summary(player)
    show_extra_statistics(player)
    show_difficulty_statistics(player)
def view_history(player):
    player.show_history()
def handle_menu_choice(choice, player):
    if choice == "1":
        play_round(player)
        return True
    elif choice == "2":
        view_statistics(player)
        return True
    elif choice == "3":
        view_history(player)
        return True
    elif choice == "4":
        show_instructions()
        return True
    elif choice == "5":
        print("Thank you for playing!")
        return False
    else:
        print("That menu option is not available.")
        return True
#SECTION 9: PROGRAM START
def main():
    show_header()
    player_name = get_player_name()
    print("Welcome to the game,", player_name + "!")
    print("Your player statistics will be recorded during this session.")
    keep_running = True
    while keep_running:
        choice = show_menu()
        if choice not in VALID_MENU_CHOICES:
            print("Invalid choice. Please enter a number from 1 to 5.")
            continue
        keep_running = handle_menu_choice(choice, player_name)
    print()
    print("FINAL SESSION SUMMARY")
    player_name.show_summary()
    print("A text copy of each round may be found in", RESULTS_FILE)
    print("Goodbye!")
if __name__ == "__main__":
    main()


                 GUESS THE NUMBER                 
A number guessing game

Welcome to the game, Brooke!
Your player statistics will be recorded during this session.

MAIN MENU
1. Play a game
2. View player statistics
3. View game history
4. Read instructions
5. Quit

DIFFICULTY LEVELS
1. Easy: numbers 1-20, 8 attempts
2. Medium: numbers 1-50, 7 attempts
3. Hard: numbers 1-100, 6 attempts

Round Start
----------------------------------------
Difficulty: Medium
The number is between 1 and 50
You have 7 attempts
Type HINT to receive a clue or QUIT to end the current game.
----------------------------------------
Attempts remaining: 7
Too high! Try a lower number.
Attempts remaining: 6
Too high! Try a lower number.
Attempts remaining: 5
Too low! Try a higher number.
Attempts remaining: 4
Too high! Try a lower number.
Attempts remaining: 3
Too low! Try a higher number.
Attempts remaining: 2
Too low! Try a higher number.
Attempts remaining: 1
Correct! You found the secret number.


AttributeError: 'str' object has no attribute 'record_result'